In [ ]:
import os, pathlib, json, math, textwrap, subprocess, wave, shutil
from PIL import Image,ImageFont,ImageDraw,ImageFilter,ImageEnhance
INPUT=pathlib.Path('/kaggle/input')
matching=list(INPUT.rglob('speech_00.wav'))
assert matching, 'Missing private dataset audio in /kaggle/input'
SRC=matching[0].parent
OUT=pathlib.Path('/kaggle/working/documentary')
OUT.mkdir(exist_ok=True)
SCENES=json.loads((SRC/'script.json').read_text())
assert len(SCENES)==8, len(SCENES)
for i in range(8):
 assert (SRC/f'scene_{i:02d}.jpg').exists()
 assert (SRC/f'speech_{i:02d}.wav').exists()
assert (SRC/'narration.wav').exists()
print('Asset source:',SRC,'NARRATION', (SRC/'narration.wav').stat().st_size,'bytes',flush=True)
print('ffmpeg',shutil.which('ffmpeg'),flush=True)

MOTION_FOUND=list(INPUT.rglob('motion_00.mp4'))
assert MOTION_FOUND, 'Missing private LTX motion clips dataset'
MOTION=MOTION_FOUND[0].parent
assert all((MOTION/f'motion_{i:02d}.mp4').exists() for i in range(8))
print('AI_MOTION_SOURCE',MOTION,flush=True)


In [ ]:
W,H,FPS=720,1280,24
def load_font(size,bold=False):
 candidates=[
  '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf' if bold else '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
  '/usr/share/fonts/truetype/liberation2/LiberationSans-Bold.ttf' if bold else '/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf',
  '/usr/share/fonts/truetype/freefont/FreeSansBold.ttf' if bold else '/usr/share/fonts/truetype/freefont/FreeSans.ttf']
 for p in candidates:
  try:return ImageFont.truetype(p,size)
  except OSError:pass
 try:return ImageFont.truetype('DejaVuSans.ttf',size)
 except OSError:
  try:return ImageFont.load_default(size=size)
  except TypeError:return ImageFont.load_default()
def fitcover(im,w,h):
 factor=max(w/im.width,h/im.height)
 nw=math.ceil(im.width*factor);nh=math.ceil(im.height*factor)
 s=im.resize((nw,nh),Image.Resampling.LANCZOS)
 return s.crop(((nw-w)//2,(nh-h)//2,(nw+w)//2,(nh+h)//2))
def wrap(draw,txt,font,maxwidth):
 words=txt.split(' ');lines=[];line=''
 for w in words:
  test=(line+' '+w).strip()
  if line and draw.textbbox((0,0),test,font=font)[2]>maxwidth:
   lines.append(line);line=w
  else:line=test
 if line:lines.append(line)
 return lines
for i,row in enumerate(SCENES):
 im=Image.open(SRC/f'scene_{i:02d}.jpg').convert('RGB')
 base=fitcover(im,W,H).filter(ImageFilter.GaussianBlur(26)).convert('RGBA')
 overlay=Image.new('RGBA',(W,H),(4,12,29,137))
 base=Image.alpha_composite(base,overlay)
 focus=fitcover(im,696,735)
 base.paste(focus,(12,215))
 draw=ImageDraw.Draw(base,'RGBA')
 draw.rectangle((0,0,W,215),fill=(3,12,28,205))
 draw.rectangle((0,875,W,H),fill=(4,12,25,223))
 head=load_font(38,True);cap=load_font(30,True);detail=load_font(20)
 draw.text((33,44),'WHAT HAPPENS',font=head,fill=(255,251,240,255))
 draw.text((33,94),'WHEN WE DIE?',font=head,fill=(255,209,125,255))
 draw.text((35,164),f'SCIENCE DOCUMENTARY  •  SCENE {i+1:02d}/08',font=detail,fill=(175,198,222,255))
 lines=wrap(draw,row['text'],cap,W-72)
 if len(lines)>7:
  cap=load_font(27,True);lines=wrap(draw,row['text'],cap,W-72)
 hline=41 if len(lines)<=7 else 36
 start=max(905,1110-len(lines)*hline//2)
 for j,line in enumerate(lines):
  draw.text((33,start+j*hline),line,font=cap,fill=(255,250,238,255),stroke_width=1,stroke_fill=(7,12,20,255))
 draw.text((35,H-63),'VOXCPM2  •  CLOUDFLARE AI  •  KAGGLE',font=detail,fill=(172,186,208,255))
 path=OUT/f'card_{i:02d}.jpg'
 base.convert('RGB').save(path,quality=92,optimize=True)
 print('CARD',i+1,path.stat().st_size,'bytes',flush=True)

# Transparent captions over AI video instead of replacing it with still images.
for i,row in enumerate(SCENES):
    overlay=Image.new('RGBA',(W,H),(0,0,0,0))
    draw=ImageDraw.Draw(overlay,'RGBA')
    draw.rectangle((0,0,W,214),fill=(3,12,28,211))
    draw.rectangle((0,876,W,H),fill=(3,12,28,215))
    head=load_font(38,True)
    cap=load_font(29,True)
    detail=load_font(20)
    draw.text((33,44),'WHAT HAPPENS',font=head,fill=(255,251,240,255))
    draw.text((33,94),'WHEN WE DIE?',font=head,fill=(255,209,125,255))
    draw.text((35,164),f'SCIENCE DOCUMENTARY  •  SCENE {i+1:02d}/08',font=detail,fill=(175,198,222,255))
    lines=wrap(draw,row['text'],cap,W-72)
    if len(lines)>7:
        cap=load_font(26,True);lines=wrap(draw,row['text'],cap,W-72)
    hline=40 if len(lines)<=7 else 35
    start=max(904,1110-len(lines)*hline//2)
    for j,line in enumerate(lines):
        draw.text((33,start+j*hline),line,font=cap,fill=(255,250,238,255),stroke_width=1,stroke_fill=(7,12,20,255))
    draw.text((35,H-63),'VOXCPM2  •  CLOUDFLARE AI  •  LTX',font=detail,fill=(172,186,208,255))
    overlay.save(OUT/f'overlay_{i:02d}.png')
print('TRANSPARENT_OVERLAYS_READY 8',flush=True)


In [ ]:

clips=[]
for i,row in enumerate(SCENES):
    sec=float(row['duration'])
    n=math.ceil(sec*FPS)
    source=MOTION/f'motion_{i:02d}.mp4'
    overlay=OUT/f'overlay_{i:02d}.png'
    dest=OUT/f'clip_{i:02d}.mp4'
    filt=(f"[0:v]scale={W}:{H}:force_original_aspect_ratio=increase,crop={W}:{H},"
           f"fps={FPS},format=yuv420p[base];[base][1:v]overlay=0:0:shortest=1,"
           f"format=yuv420p[out]")
    cmd=['ffmpeg','-y','-hide_banner','-loglevel','error',
         '-stream_loop','-1','-i',str(source),
         '-loop','1','-i',str(overlay),
         '-filter_complex',filt,'-map','[out]','-frames:v',str(n),
         '-c:v','libx264','-preset','veryfast','-crf','23',
         '-pix_fmt','yuv420p','-r',str(FPS),str(dest)]
    subprocess.run(cmd,check=True)
    assert dest.stat().st_size>15000,('bad video',dest)
    clips.append(dest)
    print('AI_MOTION_ASSEMBLED',i+1,'of',len(SCENES),'duration',sec,'size',dest.stat().st_size,flush=True)
listfile=OUT/'concat.txt'
listfile.write_text(''.join("file '"+str(x)+"'\n" for x in clips))
subprocess.run(['ffmpeg','-y','-hide_banner','-loglevel','error',
 '-f','concat','-safe','0','-i',str(listfile),'-c','copy',str(OUT/'video_silent.mp4')],check=True)
print('AI_MOTION_CONCATENATED',flush=True)


In [ ]:
import wave, numpy as np
with wave.open(str(SRC/'narration.wav'),'rb') as w:
 assert w.getnchannels()==1 and w.getsampwidth()==2
 SR=w.getframerate()
 signal=np.frombuffer(w.readframes(w.getnframes()),dtype='<i2').astype(np.float32)/32768
length=len(signal)
t=np.arange(length,dtype=np.float64)/SR
duration=length/SR
music=np.zeros(length,dtype=np.float64)
chords=((146.83,196,246.94),(174.61,220,261.63),(130.81,164.81,196))
for k,notes in enumerate(chords):
 lo=k*duration/3;hi=(k+1)*duration/3
 env=np.clip((t-lo)/1.7,0,1)*np.clip((hi-t)/1.7,0,1)
 for j,f in enumerate(notes):
  music+=(.023/(j+1))*env*np.sin(2*np.pi*f*t)
music*=np.clip(t/1.5,0,1)*np.clip((duration-t)/1.2,0,1)
combined=np.clip(.94*signal+music,-.98,.98).astype(np.float32)
import soundfile as sf
sf.write(OUT/'mixed_audio.wav',combined,SR,subtype='PCM_16')
final=OUT/'what_happens_when_we_die_AI_animated.mp4'
subprocess.run(['ffmpeg','-y','-hide_banner','-loglevel','error','-i',str(OUT/'video_silent.mp4'),
                '-i',str(OUT/'mixed_audio.wav'),'-c:v','copy','-c:a','aac','-b:a','160k',
                '-shortest','-movflags','+faststart',str(final)],check=True)
p=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size',
                   '-of','json',str(final)],check=True,capture_output=True,text=True)
meta=json.loads(p.stdout)['format']
assert float(meta['duration'])>50 and int(meta['size'])>100000
print('SUCCESS',final,'DURATION',meta['duration'],'SIZE',meta['size'],flush=True)

import shutil
shutil.copy2(MOTION/'generation_report.json',OUT/'generation_report.json')
print('AI_MOTION_FINAL_SUCCESS',str(final),int(meta['size']),meta['duration'],flush=True)
